In [5]:
import cv2
import numpy as np
import os
import sys
sys.path.append(os.path.dirname(os.path.abspath('.')))
import tqdm

video_path = '../video'
video_path_list = [os.path.join(video_path, f) for f in os.listdir(video_path) if f.endswith('.mp4')]
print(video_path_list)
# video_path_list = [video_path_list[5]]

path = '../photos/single_camera'
# path = '../photos/multi_camera'
# path = '../photos/other'

['../video/20260330_204016.mp4', '../video/20260330_203636.mp4', '../video/20260330_195942.mp4', '../video/20260330_202819.mp4']


In [4]:
if os.path.exists(f'{path}'):
    for image in os.listdir(f'{path}'):
        os.remove(f'{path}/{image}')
else:
    os.makedirs(f"{path}", exist_ok=True)

In [ ]:
os.makedirs(f"{path}", exist_ok=True)

for video_path in video_path_list:
    print(video_path)
    time_stamp = 0
    cap = cv2.VideoCapture(video_path)

    capture_per_frame = 1
    count = 0
    
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    for _ in tqdm.tqdm(range(total_frames)):
        ret, frame = cap.read()

        size = frame.shape
        # frame = frame[:size[0]//3,:]
        # frame = frame[size[0]//3*2:,:]

        if not ret:
            break
        if count % capture_per_frame == 0:
            cv2.imwrite(f"{path}/{video_path.split('/')[-1].split('.')[0]}{time_stamp}.jpg", frame)
        time_stamp += 1
        count += 1

    cap.release()

../video/20251220_172745.mp4


100%|██████████| 1697/1697 [00:17<00:00, 97.37it/s] 


## Extract Calibration Points — ChArUco Board Config

In [ ]:
from get_points import get_points_charuco

charuco_config = dict(
    charuco_squares_x=11,
    charuco_squares_y=8,
    square_size=0.023,       # metres
    marker_size=0.017,       # metres
    aruco_dict_name="DICT_5X5_100",
)


In [ ]:
### Option A — Single-camera intrinsic calibration points

intrinsic_output_dir = '../intrinsic'
intrinsic_output_file = os.path.join(intrinsic_output_dir, 'calibration_points_charuco.npz')

result_intrinsic = get_points_charuco(
    images_folder=path,
    output_file=intrinsic_output_file,
    min_corners=15,
    **charuco_config,
)
print(f"\nIntrinsic .npz saved to: {os.path.abspath(intrinsic_output_file)}")

In [ ]:
### Option B — Multi-camera extrinsic calibration points
Each image is assumed to contain camera views **stacked vertically** (top = first camera).
For single-camera extrinsic, just pass one camera name.

In [ ]:
camera_names = ["cam2", "cam1", "cam0"]     # top-to-bottom order in stacked image

extrinsic_output_dir = '../extrinsic'
extrinsic_output_file = os.path.join(extrinsic_output_dir, 'extrinsic_points_charuco.npz')

result_extrinsic = get_points_charuco(
    images_folder=path,
    output_file=extrinsic_output_file,
    camera_names=camera_names,
    min_corners=6,
    **charuco_config,
)
print(f"\nExtrinsic .npz saved to: {os.path.abspath(extrinsic_output_file)}")